# Base vs Fine-Tuned Evaluation

Training only matters if evaluation reveals what changed. This notebook compares the base model and saved LoRA adapter on held-out support questions, then interprets local metrics and failures.

A useful evaluation combines three perspectives: lexical overlap with a reference answer, probability assigned to desired answers, and direct inspection of policy behavior. No single metric is sufficient on its own.


## Learning Objectives

By the end of this notebook, you should be able to:

- Reload the unchanged base model and the saved adapter for a controlled comparison.
- Measure held-out answers with ROUGE, BLEU, and conditional perplexity.
- Inspect policy errors and explain why metric improvements do not prove reliability.


## Evaluation Plan

```mermaid
flowchart LR
    A["Held-out cases"] --> B["Base model answers"]
    A --> C["Base + LoRA answers"]
    B --> D["Lexical metrics"]
    C --> D
    B --> E["Conditional perplexity"]
    C --> E
    D --> F["Error analysis"]
    E --> F
```

ROUGE and BLEU measure lexical overlap. Perplexity measures how surprised a model is by text. These are useful signals, not final proof of answer quality.


In [ ]:
import hashlib
import json
import os
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sacrebleu
import torch
from peft import PeftModel
from rouge_score import rouge_scorer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
MODEL_REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"
ADAPTER_DIR = Path("outputs") / "lora-support-adapter"
DEVICE = torch.device("cpu")
SYSTEM_MESSAGE = "You are a concise support assistant."
MAX_NEW_TOKENS = 48

In [ ]:
evaluation_cases = json.loads(Path("data/evaluation_cases.json").read_text())
reference_material = json.loads(Path("data/reference_material.json").read_text())
reference_by_id = {item["id"]: item for item in reference_material}
evaluation_cases[0]

## Load Base and Adapter Models

If the adapter folder is missing, run notebook 03 first. The base model remains unchanged; the adapted model is the base model plus LoRA weights.


In [ ]:
if not ADAPTER_DIR.exists():
    raise FileNotFoundError(
        "Run 03-local-lora-fine-tuning-run.ipynb before this notebook. "
        f"Expected adapter folder: {ADAPTER_DIR}"
    )

training_summary = json.loads((ADAPTER_DIR / "training_summary.json").read_text())
training_hash = hashlib.sha256(
    Path("data/support_instruction_examples.jsonl").read_bytes()
).hexdigest()
if training_summary.get("training_sha256") != training_hash:
    raise ValueError(
        "Training data changed or its fingerprint is missing; rerun notebook 03."
    )

if (training_summary["base_model"], training_summary["revision"]) != (
    MODEL_NAME,
    MODEL_REVISION,
):
    raise ValueError("Adapter/base checkpoint mismatch; rerun notebook 03.")

if (
    training_summary.get("prompt_format") != "native-chat"
    or training_summary.get("system_message") != SYSTEM_MESSAGE
):
    raise ValueError("Adapter prompt format changed; rerun notebook 03.")

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
base_model.config.pad_token_id = tokenizer.eos_token_id
base_model.eval()

adapted_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
adapted_model = PeftModel.from_pretrained(adapted_base, ADAPTER_DIR)
adapted_model.config.pad_token_id = tokenizer.eos_token_id
adapted_model.eval()

In [ ]:
def build_prompt(instruction: str) -> str:
    # Match notebook 03's native chat roles and assistant-turn marker.
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": instruction},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_answer(model_to_use, instruction: str) -> tuple[str, bool]:
    # Deterministic generation keeps the comparison reproducible.
    encoded = tokenizer(
        build_prompt(instruction), return_tensors="pt", add_special_tokens=False
    ).to(DEVICE)
    generated = model_to_use.generate(
        **encoded,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = generated[0, encoded["input_ids"].shape[1] :]
    # Hitting the budget without an end token is not a completed answer.
    truncated = (
        len(new_tokens) == MAX_NEW_TOKENS
        and int(new_tokens[-1]) != tokenizer.eos_token_id
    )
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), truncated

### Exercise: Build One Prompt

Create the prompt string for the first evaluation instruction.


In [ ]:
# @TODO: Use the helper defined above on `evaluation_cases[0]`.


<details>
<summary>Show Answer</summary>

```python
example_prompt = build_prompt(evaluation_cases[0]["instruction"])
print(example_prompt)
```

</details>


## Generate Comparison Answers

The base and adapted models receive exactly the same prompt and deterministic decoding settings. This controls the comparison: any output difference comes from the adapter rather than sampling randomness.

The dataset is still far too small for production use. Treat better wording as evidence that the training signal was learned, then inspect every policy claim before concluding that the model improved.

The `*_truncated` flags identify answers cut off at the generation limit. Increase `MAX_NEW_TOKENS` and rerun this notebook before attributing missing content to model quality.


In [ ]:
rows: list[dict[str, object]] = []
# Generate both answers before calculating any metric.
for case in evaluation_cases:
    instruction = str(case["instruction"])
    base_answer, base_truncated = generate_answer(base_model, instruction)
    adapted_answer, adapted_truncated = generate_answer(adapted_model, instruction)
    rows.append(
        {
            "id": str(case["id"]),
            "instruction": instruction,
            "ideal_answer": str(case["ideal_answer"]),
            "base_answer": base_answer,
            "adapted_answer": adapted_answer,
            "base_truncated": base_truncated,
            "adapted_truncated": adapted_truncated,
        }
    )
rows[0]

## Local Metrics

ROUGE-L focuses on longest common subsequence overlap. BLEU focuses on n-gram precision. They are quick to compute, but they can miss good paraphrases and reward answers that copy reference wording while changing an important fact.

For example, “refundable until 24 hours before” and “refundable within 24 hours” share many words but express opposite policies. Metric scores must therefore be paired with semantic review.


In [ ]:
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def as_float(value: object) -> float:
    # Convert dynamic metric-table values only after checking their runtime type.
    if isinstance(value, (int, float)):
        return float(value)
    raise TypeError(f"Expected a numeric value, got {type(value).__name__}")


def rouge_l(candidate: str, reference: str) -> float:
    # Return ROUGE-L F1 for one candidate/reference pair.
    return float(rouge.score(reference, candidate)["rougeL"].fmeasure)


def bleu(candidate: str, reference: str) -> float:
    # Return sentence BLEU on the familiar 0 to 100 scale.
    return float(
        sacrebleu.sentence_bleu(candidate, [reference], use_effective_order=True).score
    )

In [ ]:
for row in rows:
    # Score the base and adapted answer against the same ideal answer.
    base_answer = str(row["base_answer"])
    adapted_answer = str(row["adapted_answer"])
    ideal_answer = str(row["ideal_answer"])
    row["base_rouge_l"] = rouge_l(base_answer, ideal_answer)
    row["adapted_rouge_l"] = rouge_l(adapted_answer, ideal_answer)
    row["base_bleu"] = bleu(base_answer, ideal_answer)
    row["adapted_bleu"] = bleu(adapted_answer, ideal_answer)

[
    {
        "id": str(row["id"]),
        "base_rouge_l": round(as_float(row["base_rouge_l"]), 3),
        "adapted_rouge_l": round(as_float(row["adapted_rouge_l"]), 3),
        "base_bleu": round(as_float(row["base_bleu"]), 2),
        "adapted_bleu": round(as_float(row["adapted_bleu"]), 2),
    }
    for row in rows
]

### Exercise: Pick the Better ROUGE-L Answer

For each row, add `rouge_winner` as `base`, `adapted`, or `tie`.


In [ ]:
# @TODO: Add a `rouge_winner` field to each row.


<details>
<summary>Show Answer</summary>

```python
for row in rows:
    # Store a per-case winner without hiding ties.
    base_rouge = as_float(row["base_rouge_l"])
    adapted_rouge = as_float(row["adapted_rouge_l"])
    if base_rouge > adapted_rouge:
        row["rouge_winner"] = "base"
    elif adapted_rouge > base_rouge:
        row["rouge_winner"] = "adapted"
    else:
        row["rouge_winner"] = "tie"

[(str(row["id"]), str(row["rouge_winner"])) for row in rows]
```

</details>


### Aggregate the Held-Out Results

Per-case results reveal failures, while averages provide a compact directional summary. With only four cases, the averages are descriptive rather than statistically reliable.


In [ ]:
metric_summary = {
    "base_mean_rouge_l": round(
        sum(as_float(row["base_rouge_l"]) for row in rows) / len(rows), 3
    ),
    "adapted_mean_rouge_l": round(
        sum(as_float(row["adapted_rouge_l"]) for row in rows) / len(rows), 3
    ),
    "base_mean_bleu": round(
        sum(as_float(row["base_bleu"]) for row in rows) / len(rows), 2
    ),
    "adapted_mean_bleu": round(
        sum(as_float(row["adapted_bleu"]) for row in rows) / len(rows), 2
    ),
}
metric_summary

### Exercise: Interpret a Metric Disagreement

Suppose adapted ROUGE improves while BLEU stays flat. What is the safest conclusion?


<details>
<summary>Show Answer</summary>

The adapted answers may preserve longer reference sequences without matching many exact n-grams. That is a signal worth investigating, not proof of better quality; inspect the generated claims and use the faithfulness checks in notebook 05.

</details>


## Perplexity on Ideal Answers

Perplexity asks how likely the ideal answer is under the model when conditioned on the instruction. Lower is better, but absolute values from small models are mainly diagnostic. We report the arithmetic mean of per-case perplexities, not corpus perplexity (which would exponentiate the token-weighted mean loss). The next cell saves all per-case metrics, aggregate scores, and training/generation settings for notebook 05.


In [ ]:
def conditional_perplexity(model_to_score, instruction: str, answer: str) -> float:
    # Use the same explicit token boundary as the training dataset.
    prompt_ids = tokenizer.encode(build_prompt(instruction), add_special_tokens=False)
    answer_ids = tokenizer.encode(answer, add_special_tokens=False)
    if not answer_ids:
        raise ValueError("Perplexity needs a non-empty target answer.")
    input_ids = torch.tensor([prompt_ids + answer_ids], device=DEVICE)
    labels = input_ids.clone()
    labels[:, : len(prompt_ids)] = -100

    with torch.no_grad():
        output = model_to_score(input_ids=input_ids, labels=labels)
    return float(torch.exp(output.loss).cpu())

In [ ]:
for row in rows:
    # Condition both models on the same prompt and score only ideal-answer tokens.
    instruction = str(row["instruction"])
    ideal_answer = str(row["ideal_answer"])
    row["base_ppl"] = conditional_perplexity(base_model, instruction, ideal_answer)
    row["adapted_ppl"] = conditional_perplexity(
        adapted_model, instruction, ideal_answer
    )

per_case_perplexity = [
    (
        str(row["id"]),
        round(as_float(row["base_ppl"]), 2),
        round(as_float(row["adapted_ppl"]), 2),
    )
    for row in rows
]
perplexity_summary = {
    "base_mean_ppl": round(
        sum(as_float(row["base_ppl"]) for row in rows) / len(rows), 2
    ),
    "adapted_mean_ppl": round(
        sum(as_float(row["adapted_ppl"]) for row in rows) / len(rows), 2
    ),
}
# Persist the complete comparison only after every metric has been calculated.
comparison_report = {
    "configuration": {
        "training": training_summary,
        "adapter_config": json.loads((ADAPTER_DIR / "adapter_config.json").read_text()),
        "generation": {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False},
        "device": str(DEVICE),
        "evaluation_sha256": hashlib.sha256(
            Path("data/evaluation_cases.json").read_bytes()
        ).hexdigest(),
    },
    "summary": {**metric_summary, **perplexity_summary},
    "cases": rows,
}
comparison_path = Path("outputs/model-comparison.json")
comparison_path.parent.mkdir(parents=True, exist_ok=True)
comparison_path.write_text(json.dumps(comparison_report, indent=2), encoding="utf-8")
{"per_case": per_case_perplexity, "mean": perplexity_summary}

### Exercise: Interpret Perplexity Direction

If the adapted model has lower perplexity on ideal answers than the base model, what does that suggest?


<details>
<summary>Show Answer</summary>

It suggests the adapted model assigns higher probability to the ideal answer style or wording. It does not prove the answer is factually correct, safe, or faithful.

</details>


Metric disagreement is itself a useful result. An adapter can increase lexical overlap while assigning lower probability to some ideal answers or reversing a policy condition. That pattern suggests it learned surface style more readily than reliable domain behavior, so the next iteration should improve data coverage rather than simply train for more epochs.


## Error Analysis

Inspect failures directly. A compact rubric keeps the review consistent:

| Dimension | Question |
| --- | --- |
| Relevance | Does the answer address the requested support task? |
| Faithfulness | Are policy claims supported by the reference material? |
| Completeness | Does it include the required decision and next step? |
| Coherence | Is the answer readable and internally consistent? |

Look for empty answers, repeated tokens, missing policy constraints, reversed conditions, or invented exceptions. Check the security cases especially: a small adapter can copy the training style ("Please send...") while reversing the policy. Notebook 05 turns the same dimensions into structured judge fields and transparent local checks.


In [ ]:
for row in rows:
    print(f"CASE: {row['id']}")
    print(f"Instruction: {row['instruction']}")
    print(f"Ideal: {row['ideal_answer']}")
    print(f"Base (truncated={row['base_truncated']}): {row['base_answer']}")
    print(f"Adapted (truncated={row['adapted_truncated']}): {row['adapted_answer']}")
    print("-" * 80)

When notebook 03 saved the adapter, PEFT also wrote a model card template at `outputs/lora-support-adapter/README.md`. Its Evaluation and Results sections are the place to record these held-out scores and the failures you found, so anyone reusing the adapter sees its limits.


## Further Reading

- [ROUGE: A Package for Automatic Evaluation of Summaries](https://aclanthology.org/W04-1013/) - Lin, 2004
- [BLEU: a Method for Automatic Evaluation of Machine Translation](https://aclanthology.org/P02-1040/) - Papineni et al., 2002
- [Evaluate Documentation](https://huggingface.co/docs/evaluate/index)
- [Transformers Perplexity Guide](https://huggingface.co/docs/transformers/perplexity)